# First Simple CNN Model

Start with the simplest next model that can plausibly beat logistic + MFCC:

- a small CNN on 2D audio representations of each breathing cycle

Plan:

**1. Fix the modelling unit**

- One row = one cycle_filename
- One label = disease for that cycle’s patient


**2. Turn each cycle .wav into a fixed-size 2D input**

Use a mel spectrogram as a 2D array:

- x-axis = time
- y-axis = frequency bands
- values = energy

**3. Build a very small CNN first**

- 2 convolution layers
- max pooling
- flatten
- dense
- softmax output

## Imports

In [ ]:
import os
from pathlib import Path
import sys

# Move notebook working directory to the project root
os.chdir("/Users/keira/code/mi-mi-mia/smart-stethoscope")

# Make sure Python can import the package
sys.path.insert(0, str(Path.cwd()))

print("Current working directory:", Path.cwd())
print("raw_data exists:", Path("raw_data").exists())
print("preprocessed_data exists:", Path("preprocessed_data").exists())

In [ ]:

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical


from smart_stethoscope.ml_logic.data_loading import load_data
from smart_stethoscope.ml_logic.preprocessing import preprocess_tabular_data

## 0. Preprocessing

In [ ]:
# Load the full dataframe
df = load_data()

print("Raw df shape:", df.shape)
print(df.columns.tolist())

In [ ]:
# 2. Run preprocessing to get split outputs
X_train_tab, X_test_tab, y_train, y_test, train_cycle_filenames, test_cycle_filenames = preprocess_tabular_data(df)

print("X_train_tab shape:", X_train_tab.shape)
print("X_test_tab shape:", X_test_tab.shape)
print("Number of train cycle filenames:", len(train_cycle_filenames))
print("Number of test cycle filenames:", len(test_cycle_filenames))

## 1. Convert one cycle into a 2D array

In [ ]:
# Define a function to transform wav file to a mel spectrogram

def wav_to_mel_spec(
    cycle_filename,
    audio_folder,
    n_mels=64,
    max_time_steps=200
):

    """
    Convert one breathing cycle .wav file into a fixed-size mel spectrogram.

    Parameters
    ----------
    cycle_filename : str
        Breathing cycle filename without the .wav extension.
        Example: '101_1b1_Al_sc_Meditron_0'
    audio_folder : str or Path
        Folder containing extracted breathing cycle .wav files.
    n_mels : int, default=64
        Number of mel frequency bins.
    max_time_steps : int, default=200
        Fixed width for the spectrogram.
        If spectrogram is shorter, pad with zeros.
        If longer, crop.

    Returns
    -------
    mel_spec_db : np.ndarray
        2D array of shape (n_mels, max_time_steps)
    """

    # Build full file path
    file_path = Path(audio_folder) / f"{cycle_filename}.wav"

    # Load audio at original sample rate
    signal, sr = librosa.load(file_path, sr=None)

    # Convert waveform to mel spectrogram (power scale)
    mel_spec = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=n_mels
    )

    # Convert to decibel scale for better modelling / visualisation
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

    # Current shape is (n_mels, time_steps)
    current_time_steps = mel_spec_db.shape[1]

    # Pad with zeros on the right if too short
    if current_time_steps < max_time_steps:
        pad_width = max_time_steps - current_time_steps
        mel_spec_db = np.pad(
            mel_spec_db,
            pad_width=((0, 0), (0, pad_width)),
            mode="constant"
        )

    # Crop on the right if too long
    elif current_time_steps > max_time_steps:
        mel_spec_db = mel_spec_db[:, :max_time_steps]

    return mel_spec_db

In [ ]:
# Select one example cycle from the train split
example_cycle = train_cycle_filenames[0]
print("Example cycle filename:", example_cycle)

In [ ]:
# Convert that one cycle into a mel spectrogram
cycle_audio_path = "preprocessed_data/audio_breathing_cycles"

example_mel = wav_to_mel_spec(
    cycle_filename=example_cycle,
    audio_folder=cycle_audio_path,
    n_mels=64,
    max_time_steps=200
)

print("Mel spectrogram shape:", example_mel.shape)

In [ ]:
# Plot the spec
plt.figure(figsize=(10, 4))
librosa.display.specshow(
    example_mel,
    x_axis="time",
    y_axis="mel"
)
plt.colorbar(format="%+2.0f dB")
plt.title("Example breathing cycle mel spectrogram")
plt.tight_layout()
plt.show()

## 2. Convert all train and test cycles

In [ ]:
# Assumes these already exist from previous steps:
# - train_cycle_filenames
# - test_cycle_filenames
# - wav_to_mel_spec()
# - cycle_audio_path

def build_mel_dataset(
    cycle_filenames,
    audio_folder,
    n_mels=64,
    max_time_steps=200):
    """
    Convert a list/array of cycle filenames into a 4D tensor for CNN input.

    Parameters
    ----------
    cycle_filenames : iterable of str
        Cycle filenames without .wav extension
    audio_folder : str or Path
        Folder containing cycle .wav files
    n_mels : int, default=64
        Number of mel bins
    max_time_steps : int, default=200
        Fixed spectrogram width

    Returns
    -------
    X : np.ndarray
        CNN-ready tensor of shape (n_samples, n_mels, max_time_steps, 1)
    """

    mel_specs = []

    for cycle_filename in cycle_filenames:
        # convert each wav to a mel spec
        mel = wav_to_mel_spec(
            cycle_filename=cycle_filename,
            audio_folder=audio_folder,
            n_mels=n_mels,
            max_time_steps=max_time_steps
        )
        # add each 2D array to the list
        mel_specs.append(mel)

    # Stack into one 3D array: (n_samples, n_mels, max_time_steps)
    X = np.stack(mel_specs)

    # Add channel dimension for expected CNN shape: (n_samples, n_mels, max_time_steps, 1)
    X = X[..., np.newaxis]

    return X

In [ ]:
# Build train and test image tensors
cycle_audio_path = "preprocessed_data/audio_breathing_cycles"

X_train_img = build_mel_dataset(
    cycle_filenames=train_cycle_filenames,
    audio_folder=cycle_audio_path,
    n_mels=64,
    max_time_steps=200
)

X_test_img = build_mel_dataset(
    cycle_filenames=test_cycle_filenames,
    audio_folder=cycle_audio_path,
    n_mels=64,
    max_time_steps=200
)

print("X_train_img shape:", X_train_img.shape)
print("X_test_img shape:", X_test_img.shape)
print("Expected CNN input shape per sample:", X_train_img[0].shape)

## 3. Encode labels for CNN

In [ ]:
# Check shape of data
print("y_train shape before encoding:", y_train.shape)
print("y_test shape before encoding:", y_test.shape)

# Check class labels
print("Unique train labels:", sorted(y_train.unique()))
print("Unique test labels:", sorted(y_test.unique()))

# Calculate number of classes
num_classes = len(np.unique(y_train))
print("Number of classes:", num_classes)

In [ ]:
# Convert integer labels to one-hot encoded arrays
# Example: class 2 becomes [0, 0, 1, 0, 0, 0]
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

# Check the new shapes
print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

# Look at one example
print("First raw label:", y_train.iloc[0])
print("First one-hot label:", y_train_cat[0])

## 4. Build a very small CNN

Build a small CNN for multi-class breathing cycle classification:

In [ ]:
# Check the shape of one training example
# Expected shape per sample = (64, 200, 1)
# 64 mel bins (height), 200 time steps (width), 1 channel (like a grayscale image)
print("Single training example shape:", X_train_img[0].shape)

# Work out the input shape for the CNN
# Keras wants the shape of ONE sample, not the full dataset
input_shape = X_train_img.shape[1:]

print("CNN input shape:", input_shape)
print("Number of output classes:", num_classes)

In [ ]:
# Build a small sequential CNN

# ------------------------------------------------
# Architecture:
# 1. Convolution layer
# 2. Max pooling
# 3. Convolution layer
# 4. Max pooling
# 5. Flatten
# 6. Dense hidden layer
# 7. Softmax output layer
# ------------------------------------------------

cnn_model = models.Sequential()

# First convolution block
cnn_model.add(layers.Input(shape=input_shape))
cnn_model.add(
    layers.Conv2D(
        filters=16,                 # number of filters / feature detectors
        kernel_size=(3, 3),         # small 2D window scanning the image
        activation="relu",          # non-linear activation
        padding="same"              # keep output width/height similar
    )
)
cnn_model.add(layers.MaxPooling2D(pool_size=(2, 2)))  # reduce spatial size

# Second convolution block
cnn_model.add(layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    )
)
cnn_model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Flatten 2D feature maps into 1D vector
cnn_model.add(layers.GlobalAveragePooling2D())

# Dense hidden layer
cnn_model.add(layers.Dense(32,activation="relu"))

# Dropout to reduce overfitting a bit
cnn_model.add(layers.Dropout(0.3))

# Output layer
# num_classes = 6, so softmax returns 6 probabilities
cnn_model.add(layers.Dense(num_classes, activation="softmax"))

# Compile the model
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

## 5. Train and compare performance

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Train the model
history = cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate on test set
test_loss, test_accuracy = cnn_model.evaluate(X_test_img, y_test_cat, verbose=1)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

Get predictions and inspect performance:

In [ ]:
# Predict class probabilities
y_pred_probs = cnn_model.predict(X_test_img)

# Convert probabilities to class labels
y_pred_cnn = np.argmax(y_pred_probs, axis=1)

# Convert one-hot encoded y_test back to integer labels
y_test_int = np.argmax(y_test_cat, axis=1)

print(classification_report(y_test_int, y_pred_cnn, zero_division=0))

## Iterations

### Iteration 1️⃣: CNN with class weights

In [ ]:
# Convert one-hot back to integer labels
y_train_int = np.argmax(y_train_cat, axis=1)

# Compute class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_int),
    y=y_train_int
)

class_weights_dict = dict(enumerate(class_weights))

print("Class weights:", class_weights_dict)

In [ ]:
# Train model with class weights
history = cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_accuracy = cnn_model.evaluate(X_test_img, y_test_cat, verbose=1)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

In [ ]:
# Predict and evaluate
y_pred_probs = cnn_model.predict(X_test_img)

y_pred_cnn = np.argmax(y_pred_probs, axis=1)
y_test_int = np.argmax(y_test_cat, axis=1)

print(classification_report(y_test_int, y_pred_cnn, zero_division=0))

In [ ]:
# Plot training vs validation curves

history_dict = history.history

# Loss plot
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_dict["loss"], label="Train Loss")
plt.plot(history_dict["val_loss"], label="Val Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy plot
plt.subplot(1,2,2)
plt.plot(history_dict["accuracy"], label="Train Accuracy")
plt.plot(history_dict["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

### ⭐️ Best So Far ⭐️
### Iteration 2️⃣: We need a kinder patience parameter in EarlyStopping

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5, # instead of 3
    restore_best_weights=True
)

history = cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_accuracy = cnn_model.evaluate(X_test_img, y_test_cat, verbose=1)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

# Predict and evaluate
y_pred_probs = cnn_model.predict(X_test_img)

y_pred_cnn = np.argmax(y_pred_probs, axis=1)
y_test_int = np.argmax(y_test_cat, axis=1)

print(classification_report(y_test_int, y_pred_cnn, zero_division=0))

# ⭐️ SAVE MVP MODEL
cnn_model.save("best_cnn_model.keras")

In [ ]:
# Plot training vs validation curves

history_dict = history.history

# Loss plot
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_dict["loss"], label="Train Loss")
plt.plot(history_dict["val_loss"], label="Val Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy plot
plt.subplot(1,2,2)
plt.plot(history_dict["accuracy"], label="Train Accuracy")
plt.plot(history_dict["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
print(len(history.history["loss"]))

### Iteration 3️⃣: A stronger CNN (increase filters)

In [ ]:
input_shape = X_train_img.shape[1:]

cnn_model = models.Sequential()

# First convolution block
cnn_model.add(layers.Input(shape=input_shape))
cnn_model.add(
    layers.Conv2D(
        filters=32,                 # increased from 16
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    )
)
cnn_model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Second convolution block
cnn_model.add(
    layers.Conv2D(
        filters=64,                 # increased from 32
        kernel_size=(3, 3),
        activation="relu",
        padding="same"
    )
)
cnn_model.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Global average pooling keeps parameter count low
cnn_model.add(layers.GlobalAveragePooling2D())

# Dense hidden layer
cnn_model.add(layers.Dense(32, activation="relu"))

# Dropout
cnn_model.add(layers.Dropout(0.3))

# Output layer
cnn_model.add(layers.Dense(num_classes, activation="softmax"))

# Compile model
cnn_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Show model summary
cnn_model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = cnn_model.fit(
    X_train_img,
    y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights_dict,
    callbacks=[early_stopping],
    verbose=1
)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

In [ ]:
# Predict and evaluate
y_pred_probs = cnn_model.predict(X_test_img)

y_pred_cnn = np.argmax(y_pred_probs, axis=1)
y_test_int = np.argmax(y_test_cat, axis=1)

print(classification_report(y_test_int, y_pred_cnn, zero_division=0))

In [ ]:
# Plot training vs validation curves

history_dict = history.history

# Loss plot
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_dict["loss"], label="Train Loss")
plt.plot(history_dict["val_loss"], label="Val Loss")
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy plot
plt.subplot(1,2,2)
plt.plot(history_dict["accuracy"], label="Train Accuracy")
plt.plot(history_dict["val_accuracy"], label="Val Accuracy")
plt.title("Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Recompute predictions from the CURRENT trained model
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# 1. Evaluate directly in Keras
test_loss, test_accuracy = cnn_model.evaluate(X_test_img, y_test_cat, verbose=1)
print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

# 2. Predict on test set using the SAME model
y_pred_probs = cnn_model.predict(X_test_img, verbose=1)
y_pred_cnn = np.argmax(y_pred_probs, axis=1)

# 3. Convert one-hot test labels back to integer labels
y_test_int = np.argmax(y_test_cat, axis=1)

# 4. Sanity check: sklearn accuracy should match Keras accuracy closely
print("Sklearn accuracy:", accuracy_score(y_test_int, y_pred_cnn))

# 5. Classification report
print(classification_report(y_test_int, y_pred_cnn, zero_division=0))

# 6. Optional: confusion matrix
print(confusion_matrix(y_test_int, y_pred_cnn))